In [113]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [114]:
from pathlib import Path
from datetime import date, timedelta
import csv
import math
import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn
from tqdm.auto import tqdm

PROJECT_PATH = Path("/content/drive/MyDrive/multimodal-fashion-recsys")

RAW_PATH = PROJECT_PATH / "data" / "raw"
PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
CHECKPOINTS_PATH = PROJECT_PATH / "checkpoints"
EMBEDDINGS_PATH = PROJECT_PATH / "embeddings"
SUBMISSIONS_PATH = PROJECT_PATH / "submissions"

SUBMISSIONS_PATH.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PREDICT_START = date(2020, 9, 23)

HISTORY_DAYS = 56
VISUAL_HALF_LIFE = 7
DECAY_HALF_LIFE = 3

HISTORY_WEIGHT = 1.0
SASREC_WEIGHT = 0.12
DECAY_WEIGHT = 0.08
VISUAL_WEIGHT = 0.07

TOP_N = 100
K = 12

print("Device:", DEVICE)

Device: cuda


In [116]:
import io
import zipfile

transactions = pl.scan_parquet(
    PROCESSED_PATH / "transactions_mapped.parquet"
)

customer_mapping = pl.read_parquet(
    PROCESSED_PATH / "customer_mapping.parquet"
)

article_mapping = pl.read_parquet(
    PROCESSED_PATH / "article_mapping.parquet"
)

SAMPLE_PATH = RAW_PATH / "sample_submission.csv"
ZIP_PATH = RAW_PATH / "h-and-m-personalized-fashion-recommendations.zip"

if SAMPLE_PATH.exists():
    sample_submission = pl.read_csv(
        SAMPLE_PATH,
        schema_overrides={"customer_id": pl.String}
    )
else:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        sample_submission = pl.read_csv(
            io.BytesIO(
                archive.read("sample_submission.csv")
            ),
            schema_overrides={"customer_id": pl.String}
        )

NUM_ITEMS = article_mapping.height + 1

print("Customers:", customer_mapping.height)
print("Items:", NUM_ITEMS - 1)
print("Submission rows:", sample_submission.height)

Customers: 1371980
Items: 105542
Submission rows: 1371980


In [117]:
class SASRec(nn.Module):
    def __init__(
        self,
        num_items,
        max_len,
        hidden_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.item_embedding = nn.Embedding(
            num_items,
            hidden_dim,
            padding_idx=0
        )

        self.position_embedding = nn.Embedding(
            max_len,
            hidden_dim
        )

        self.dropout = nn.Dropout(dropout)

        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            layer,
            num_layers=num_layers,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, input_items):
        seq_len = input_items.size(1)

        positions = torch.arange(
            seq_len,
            device=input_items.device
        ).unsqueeze(0)

        x = self.item_embedding(input_items)
        x = x * math.sqrt(self.hidden_dim)

        x = x + self.position_embedding(positions)
        x = self.dropout(x)

        padding_mask = input_items == 0

        x = x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                device=input_items.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        x = self.transformer(
            x,
            mask=causal_mask,
            src_key_padding_mask=padding_mask
        )

        x = self.norm(x)

        return x.masked_fill(
            padding_mask.unsqueeze(-1),
            0.0
        )

In [118]:
SASREC_CHECKPOINT_PATH = CHECKPOINTS_PATH / "sasrec_best.pt"

checkpoint = torch.load(
    SASREC_CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

MAX_LEN = checkpoint["max_len"]

model = SASRec(
    num_items=NUM_ITEMS,
    max_len=MAX_LEN,
    hidden_dim=checkpoint["hidden_dim"],
    num_heads=checkpoint["num_heads"],
    num_layers=checkpoint["num_layers"],
    dropout=checkpoint["dropout"]
).to(DEVICE)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Checkpoint:", SASREC_CHECKPOINT_PATH)
print("MAX_LEN:", MAX_LEN)
print("Hidden dim:", checkpoint["hidden_dim"])

Checkpoint: /content/drive/MyDrive/multimodal-fashion-recsys/checkpoints/sasrec_best.pt
MAX_LEN: 64
Hidden dim: 192


In [119]:
VISUAL_EMBEDDINGS_PATH = (
    EMBEDDINGS_PATH
    / "clip_vit_b32_embeddings.npy"
)

visual_embeddings = np.load(
    VISUAL_EMBEDDINGS_PATH,
    mmap_mode="r"
)

assert visual_embeddings.shape == (NUM_ITEMS, 512)

visual_norms = np.linalg.norm(
    visual_embeddings.astype(np.float32),
    axis=1
)

valid_visual_items = visual_norms > 0
valid_visual_items[0] = False

print("Shape:", visual_embeddings.shape)
print("Dtype:", visual_embeddings.dtype)
print("Items with visual embeddings:", valid_visual_items.sum())

Shape: (105543, 512)
Dtype: float16
Items with visual embeddings: 105100


In [120]:
assert DEVICE.type == "cuda", "Для финального inference сейчас лучше использовать T4"

visual_embeddings_gpu = torch.tensor(
    visual_embeddings,
    dtype=torch.float16,
    device=DEVICE
)

visual_embeddings_gpu = F.normalize(
    visual_embeddings_gpu,
    p=2,
    dim=1
)

valid_visual_items_gpu = torch.tensor(
    valid_visual_items,
    dtype=torch.bool,
    device=DEVICE
)

print(
    "GPU memory:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

GPU memory: 0.96 GB


In [121]:
users = (
    sample_submission
    .select("customer_id")
    .with_row_index("row_id")
    .join(
        customer_mapping,
        on="customer_id",
        how="left"
    )
    .sort("row_id")
)

assert users.height == 1371980
assert users["customer_idx"].null_count() == 0

users.head()

row_id,customer_id,customer_idx
u32,str,u32
0,"""00000dbacae5abe5e23885899a1fa4…",1
1,"""0000423b00ade91418cceaf3b26c6a…",2
2,"""000058a12d5b43e67d225668fa1f8d…",3
3,"""00005ca1c9ed5f5146b52ac8639a40…",4
4,"""00006413d8573cd20ed7128e53b7b1…",5


In [122]:
HISTORY_START = PREDICT_START - timedelta(days=HISTORY_DAYS)

recent_user_data = (
    transactions
    .filter(
        (pl.col("t_dat") >= HISTORY_START)
        & (pl.col("t_dat") < PREDICT_START)
    )
    .sort(
        ["customer_idx", "t_dat"],
        descending=[False, True]
    )
    .group_by(
        "customer_idx",
        maintain_order=True
    )
    .agg(
        pl.col("article_idx")
        .unique(maintain_order=True)
        .head(TOP_N)
        .alias("history"),

        pl.col("article_idx")
        .alias("visual_items"),

        pl.col("t_dat")
        .alias("visual_dates")
    )
    .collect()
)

print("Users with recent history:", recent_user_data.height)
recent_user_data.head()

Users with recent history: 378688


customer_idx,history,visual_items,visual_dates
u32,list[u32],list[u32],list[date]
1,[16024],[16024],[2020-09-05]
3,[78504],[78504],[2020-09-15]
5,"[77916, 60764, … 104813]","[77916, 60764, … 104813]","[2020-08-12, 2020-08-12, … 2020-08-12]"
7,"[3092, 58296]","[3092, 58296]","[2020-09-14, 2020-09-14]"
14,"[15990, 104306, … 53893]","[15990, 104306, … 53893]","[2020-08-30, 2020-08-30, … 2020-08-14]"


In [123]:
sequence_data = (
    transactions
    .filter(pl.col("t_dat") < PREDICT_START)
    .select(
        "customer_idx",
        "t_dat",
        "article_idx"
    )
    .sort(
        ["customer_idx", "t_dat"]
    )
    .group_by(
        "customer_idx",
        maintain_order=True
    )
    .agg(
        pl.col("article_idx")
        .tail(MAX_LEN)
        .alias("sequence")
    )
    .collect()
)

print("Users with sequence:", sequence_data.height)
sequence_data.head()

Users with sequence: 1362281


customer_idx,sequence
u32,list[u32]
1,"[30328, 100, … 16024]"
2,"[43107, 31387, … 87146]"
3,"[40180, 10521, … 78504]"
4,"[61176, 64526]"
5,"[43443, 32248, … 104813]"


In [124]:
user_data = (
    users
    .join(
        sequence_data,
        on="customer_idx",
        how="left"
    )
    .join(
        recent_user_data,
        on="customer_idx",
        how="left"
    )
    .sort("row_id")
)

print("Users:", user_data.height)

print(
    "With sequence:",
    user_data["sequence"].is_not_null().sum()
)

print(
    "With recent history:",
    user_data["history"].is_not_null().sum()
)

Users: 1371980
With sequence: 1362281
With recent history: 378688


In [125]:
age_days = (
    pl.lit(PREDICT_START)
    - pl.col("t_dat")
).dt.total_days()

decay_popularity = (
    transactions
    .filter(pl.col("t_dat") < PREDICT_START)
    .with_columns(
        (
            -math.log(2)
            * age_days
            / DECAY_HALF_LIFE
        )
        .exp()
        .alias("weight")
    )
    .group_by("article_idx")
    .agg(
        pl.col("weight")
        .sum()
        .alias("score")
    )
    .sort(
        "score",
        descending=True
    )
    .head(TOP_N)
    .collect()
)

decay_top100 = (
    decay_popularity["article_idx"]
    .to_numpy()
    .astype(np.int64)
)

print(decay_top100[:12])

[104554 104555 104073  67523   3092 104528 103797  95500 103109  56695
 104046  71108]


In [126]:
def build_sequence_batch(sequences):
    batch = np.zeros(
        (len(sequences), MAX_LEN),
        dtype=np.int64
    )

    valid_indices = []

    for i, sequence in enumerate(sequences):
        if sequence is None or len(sequence) == 0:
            continue

        sequence = sequence[-MAX_LEN:]

        batch[
            i,
            -len(sequence):
        ] = sequence

        valid_indices.append(i)

    return batch, valid_indices

In [127]:
def build_history_batch(histories):
    batch = np.zeros(
        (len(histories), TOP_N),
        dtype=np.int64
    )

    for i, history in enumerate(histories):
        if history is None:
            continue

        history = history[:TOP_N]

        batch[
            i,
            :len(history)
        ] = history

    return batch

In [128]:
def build_visual_profiles(items_batch, dates_batch):
    profiles = np.zeros(
        (len(items_batch), 512),
        dtype=np.float32
    )

    valid_users = []

    for i, (items, dates) in enumerate(
        zip(items_batch, dates_batch)
    ):
        if items is None or dates is None:
            continue

        ids = np.asarray(
            items,
            dtype=np.int64
        )

        valid = valid_visual_items[ids]

        if not valid.any():
            continue

        ids = ids[valid]

        selected_dates = [
            d
            for d, keep in zip(dates, valid)
            if keep
        ]

        ages = np.asarray(
            [
                (PREDICT_START - d).days
                for d in selected_dates
            ],
            dtype=np.float32
        )

        weights = np.exp(
            -math.log(2)
            * ages
            / VISUAL_HALF_LIFE
        )

        embeddings = visual_embeddings[
            ids
        ].astype(np.float32)

        profile = (
            embeddings
            * weights[:, None]
        ).sum(axis=0)

        weight_sum = weights.sum()

        if weight_sum == 0:
            continue

        profile /= weight_sum

        norm = np.linalg.norm(profile)

        if norm == 0:
            continue

        profiles[i] = profile / norm
        valid_users.append(i)

    return profiles, valid_users

In [129]:
test_sequences = user_data["sequence"].head(100).to_list()
test_histories = user_data["history"].head(100).to_list()

sequence_batch, valid_sequence_indices = build_sequence_batch(
    test_sequences
)

history_batch = build_history_batch(
    test_histories
)

assert sequence_batch.shape == (100, MAX_LEN)
assert history_batch.shape == (100, TOP_N)

print("Sequence batch:", sequence_batch.shape)
print("History batch:", history_batch.shape)
print("Valid sequences:", len(valid_sequence_indices))

Sequence batch: (100, 64)
History batch: (100, 100)
Valid sequences: 100


In [130]:
history_rank_scores = (
    HISTORY_WEIGHT
    / torch.arange(
        1,
        TOP_N + 1,
        device=DEVICE,
        dtype=torch.float32
    )
)

sasrec_rank_scores = (
    SASREC_WEIGHT
    / torch.arange(
        1,
        TOP_N + 1,
        device=DEVICE,
        dtype=torch.float32
    )
)

decay_rank_scores = (
    DECAY_WEIGHT
    / torch.arange(
        1,
        TOP_N + 1,
        device=DEVICE,
        dtype=torch.float32
    )
)

visual_rank_scores = (
    VISUAL_WEIGHT
    / torch.arange(
        1,
        TOP_N + 1,
        device=DEVICE,
        dtype=torch.float32
    )
)

decay_top100_gpu = torch.tensor(
    decay_top100,
    dtype=torch.long,
    device=DEVICE
)

print("Ready")

Ready


In [131]:
BATCH_SIZE = 256

final_predictions = np.zeros(
    (user_data.height, K),
    dtype=np.int32
)

model.eval()

for start in tqdm(
    range(0, user_data.height, BATCH_SIZE),
    desc="Final hybrid"
):
    end = min(
        start + BATCH_SIZE,
        user_data.height
    )

    batch_size = end - start

    sequences = (
        user_data["sequence"]
        .slice(start, batch_size)
        .to_list()
    )

    histories = (
        user_data["history"]
        .slice(start, batch_size)
        .to_list()
    )

    visual_items = (
        user_data["visual_items"]
        .slice(start, batch_size)
        .to_list()
    )

    visual_dates = (
        user_data["visual_dates"]
        .slice(start, batch_size)
        .to_list()
    )

    fusion_scores = torch.zeros(
        (batch_size, NUM_ITEMS),
        device=DEVICE,
        dtype=torch.float32
    )

    history_batch = build_history_batch(
        histories
    )

    history_tensor = torch.from_numpy(
        history_batch
    ).to(DEVICE)

    fusion_scores.scatter_add_(
        1,
        history_tensor,
        history_rank_scores
        .unsqueeze(0)
        .expand(batch_size, -1)
    )

    decay_indices = (
        decay_top100_gpu
        .unsqueeze(0)
        .expand(batch_size, -1)
    )

    fusion_scores.scatter_add_(
        1,
        decay_indices,
        decay_rank_scores
        .unsqueeze(0)
        .expand(batch_size, -1)
    )

    sequence_batch, valid_sequence_indices = (
        build_sequence_batch(sequences)
    )

    if valid_sequence_indices:
        valid_sequence_indices_gpu = torch.tensor(
            valid_sequence_indices,
            device=DEVICE,
            dtype=torch.long
        )

        sequence_tensor = torch.from_numpy(
            sequence_batch[
                valid_sequence_indices
            ]
        ).to(DEVICE)

        with torch.inference_mode():
            with torch.amp.autocast(
                "cuda",
                dtype=torch.float16
            ):
                hidden = model(
                    sequence_tensor
                )

                user_embeddings = hidden[:, -1]

                sasrec_scores = (
                    user_embeddings
                    @ model.item_embedding.weight.T
                )

            sasrec_scores[:, 0] = -torch.inf

            sasrec_top100 = torch.topk(
                sasrec_scores,
                TOP_N,
                dim=1
            ).indices

        fusion_scores[
            valid_sequence_indices_gpu[:, None],
            sasrec_top100
        ] += sasrec_rank_scores.unsqueeze(0)

        del (
            sequence_tensor,
            hidden,
            user_embeddings,
            sasrec_scores,
            sasrec_top100
        )

    visual_profiles, valid_visual_users = (
        build_visual_profiles(
            visual_items,
            visual_dates
        )
    )

    if valid_visual_users:
        valid_visual_users_gpu = torch.tensor(
            valid_visual_users,
            device=DEVICE,
            dtype=torch.long
        )

        profile_tensor = torch.from_numpy(
            visual_profiles[
                valid_visual_users
            ]
        ).to(
            DEVICE,
            dtype=torch.float16
        )

        with torch.inference_mode():
            visual_scores = (
                profile_tensor
                @ visual_embeddings_gpu.T
            )

            visual_scores[
                :,
                ~valid_visual_items_gpu
            ] = -torch.inf

            visual_top100 = torch.topk(
                visual_scores,
                TOP_N,
                dim=1
            ).indices

        fusion_scores[
            valid_visual_users_gpu[:, None],
            visual_top100
        ] += visual_rank_scores.unsqueeze(0)

        del (
            profile_tensor,
            visual_scores,
            visual_top100
        )

    fusion_scores[:, 0] = -torch.inf

    predictions = torch.topk(
        fusion_scores,
        K,
        dim=1
    ).indices

    final_predictions[
        start:end
    ] = predictions.cpu().numpy().astype(
        np.int32
    )

    del (
        fusion_scores,
        history_tensor,
        predictions
    )

Final hybrid:   0%|          | 0/5360 [00:00<?, ?it/s]

In [132]:
print("Shape:", final_predictions.shape)
print("First:", final_predictions[0])

assert final_predictions.shape == (
    1371980,
    12
)

assert (final_predictions > 0).all()

Shape: (1371980, 12)
First: [ 16024 104554  16025 104841 104555  67523  12402  97319  67544 104073
  16020  71108]


In [133]:
PREDICTIONS_PATH = (
    EMBEDDINGS_PATH
    / "final_kaggle_predictions.npy"
)

np.save(
    PREDICTIONS_PATH,
    final_predictions
)

print("Saved:", PREDICTIONS_PATH)
print(
    f"Size: {PREDICTIONS_PATH.stat().st_size / 1024**2:.2f} MB"
)

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/embeddings/final_kaggle_predictions.npy
Size: 62.80 MB


In [134]:
idx_to_article = np.zeros(
    NUM_ITEMS,
    dtype=np.int64
)

idx_to_article[
    article_mapping["article_idx"].to_numpy()
] = article_mapping["article_id"].to_numpy()

test_article_ids = idx_to_article[
    final_predictions[0]
]

print(test_article_ids)

[568601043 924243001 568601044 927922002 924243002 751471001 553092021
 873678003 751471043 918522001 568601033 762846027]


In [135]:
SUBMISSION_PATH = (
    SUBMISSIONS_PATH
    / "final_hybrid_submission.csv"
)

customer_ids = user_data[
    "customer_id"
].to_list()

with open(
    SUBMISSION_PATH,
    "w",
    newline=""
) as file:
    writer = csv.writer(file)

    writer.writerow([
        "customer_id",
        "prediction"
    ])

    for customer_id, prediction in tqdm(
        zip(
            customer_ids,
            final_predictions
        ),
        total=len(customer_ids),
        desc="Writing submission"
    ):
        article_ids = idx_to_article[
            prediction
        ]

        prediction_string = " ".join(
            str(int(article_id)).zfill(10)
            for article_id in article_ids
        )

        writer.writerow([
            customer_id,
            prediction_string
        ])

print("Saved:", SUBMISSION_PATH)
print(
    f"Size: {SUBMISSION_PATH.stat().st_size / 1024**2:.2f} MB"
)

Writing submission:   0%|          | 0/1371980 [00:00<?, ?it/s]

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/submissions/final_hybrid_submission.csv
Size: 259.07 MB


In [136]:
submission_check = pl.read_csv(
    SUBMISSION_PATH
)

prediction_lengths = (
    submission_check["prediction"]
    .str.split(" ")
    .list.len()
)

print("Shape:", submission_check.shape)
print(
    "Unique customers:",
    submission_check["customer_id"].n_unique()
)
print(
    "Min predictions:",
    prediction_lengths.min()
)
print(
    "Max predictions:",
    prediction_lengths.max()
)

submission_check.head()

Shape: (1371980, 2)
Unique customers: 1371980
Min predictions: 12
Max predictions: 12


customer_id,prediction
str,str
"""00000dbacae5abe5e23885899a1fa4…","""0568601043 0924243001 05686010…"
"""0000423b00ade91418cceaf3b26c6a…","""0599580038 0924243001 07174900…"
"""000058a12d5b43e67d225668fa1f8d…","""0794321007 0924243001 07514710…"
"""00005ca1c9ed5f5146b52ac8639a40…","""0730683001 0924243001 07201250…"
"""00006413d8573cd20ed7128e53b7b1…","""0791587015 0730683050 08961520…"


In [137]:
assert (
    submission_check["customer_id"]
    == sample_submission["customer_id"]
).all()

assert submission_check.height == 1371980
assert prediction_lengths.min() == 12
assert prediction_lengths.max() == 12

print("Submission is ready")

Submission is ready


In [138]:
SUBMISSIONS_PATH = PROJECT_PATH / "submissions"
SUBMISSIONS_PATH.mkdir(exist_ok=True)

SUBMISSION_PATH = SUBMISSIONS_PATH / "submission.csv"

submission_check.write_csv(SUBMISSION_PATH)

print("Saved:", SUBMISSION_PATH)
print(f"Size: {SUBMISSION_PATH.stat().st_size / 1024**2:.2f} MB")

Saved: /content/drive/MyDrive/multimodal-fashion-recsys/submissions/submission.csv
Size: 257.76 MB


In [139]:
saved_submission = pl.read_csv(
    SUBMISSION_PATH,
    schema_overrides={
        "customer_id": pl.String,
        "prediction": pl.String
    }
)

saved_lengths = (
    saved_submission["prediction"]
    .str.split(" ")
    .list.len()
)

assert saved_submission.height == 1371980
assert saved_submission["customer_id"].n_unique() == 1371980
assert saved_lengths.min() == 12
assert saved_lengths.max() == 12

print("Saved submission is valid")
saved_submission.head()

Saved submission is valid


customer_id,prediction
str,str
"""00000dbacae5abe5e23885899a1fa4…","""0568601043 0924243001 05686010…"
"""0000423b00ade91418cceaf3b26c6a…","""0599580038 0924243001 07174900…"
"""000058a12d5b43e67d225668fa1f8d…","""0794321007 0924243001 07514710…"
"""00005ca1c9ed5f5146b52ac8639a40…","""0730683001 0924243001 07201250…"
"""00006413d8573cd20ed7128e53b7b1…","""0791587015 0730683050 08961520…"


In [ ]:
from datetime import timedelta
import math

transactions = pl.scan_parquet(
    PROCESSED_PATH / "transactions_mapped.parquet"
)

MAX_DATE = (
    transactions
    .select(pl.col("t_dat").max())
    .collect()
    .item()
)

HISTORY_DAYS = 56
HALF_LIFE = 3

HISTORY_START = MAX_DATE - timedelta(days=HISTORY_DAYS)

print("Last transaction:", MAX_DATE)
print("History start:", HISTORY_START)